# Correct infection rates: cell-calling and denominators

**Utility:** an unfiltered scRNA-seq matrix is mostly empty droplets. Reporting
"% infected" over *all barcodes* dilutes the rate toward zero. ViralScan calls real
cells and reports viral rates over that biologically meaningful denominator
(`pct_infected_called`) alongside the all-barcode number.

In the manuscript (*Full-depth benchmarks*), HSV-1 looked like **0.55%** infected over
all 1.89M raw barcodes, but **13–18%** over 4,414 called cells at matched thresholds —
matching the published 13–19%. The discrepancy was entirely a denominator artifact.

## A synthetic matrix: a few real cells among many empty droplets

5 barcodes, 3 genes (`v1`,`v2` viral; `h1` host). Barcodes 0–1 are real cells with
high UMI; 2–4 are ambient/empty droplets that still carry a stray viral count.

In [ ]:
import numpy as np
import anndata as ad
import pandas as pd
import scipy.sparse as sp

X = np.array([
    [10, 0, 900],  # bc0 real cell, viral+
    [0, 5, 800],   # bc1 real cell, viral+
    [1, 0, 3],     # bc2 empty droplet, stray viral
    [0, 0, 2],     # bc3 empty droplet
    [2, 1, 1],     # bc4 empty droplet, stray viral
], dtype=float)
adata = ad.AnnData(sp.csr_matrix(X))
adata.obs_names = [f'bc{i}' for i in range(5)]
adata.var_names = ['v1', 'v2', 'h1']
adata

## Call cells with the knee method

`knee_cells(total_umi, min_umi)` keeps barcodes above a total-UMI knee. ViralScan
also supports `emptydrops` (DropletUtils, R) and `external` (supply a CellRanger /
STARsolo called-cell list via `--called-cells-file`).

In [ ]:
from viralscan.scripts.cellcalling import knee_cells

total_umi = np.asarray(adata.X.sum(axis=1)).ravel()
called_mask = knee_cells(total_umi, min_umi=100.0)
called = [bc for bc, keep in zip(adata.obs_names, called_mask) if keep]
print('total UMI per barcode:', total_umi.tolist())
print('called cells         :', called)

## Compare denominators with `compute_stats`

`compute_stats` reports both the all-barcode rate (`pct_infected`) and the
called-cell rate (`pct_infected_called`). The viral genes are grouped per virus.

In [ ]:
from viralscan.scripts.detection import compute_stats

group_by_virus = {'virusA': ['v1', 'v2']}
found_genes = {'v1': 1, 'v2': 1}

stats_all, _ = compute_stats(adata, found_genes, group_by_virus, [])
stats_called, per_cell = compute_stats(
    adata, found_genes, group_by_virus, [], called_mask=called_mask
)
s_all, s_called = stats_all['virusA'], stats_called['virusA']

pd.DataFrame({
    'metric': ['infected_cells', 'total_cells', 'pct_infected (all barcodes)',
               'n_called_cells', 'infected_called', 'pct_infected_called'],
    'value': [s_all['infected_cells'], s_all['total_cells'], s_all['pct_infected'],
              s_called['n_called_cells'], s_called['infected_called'],
              s_called['pct_infected_called']],
})

The all-barcode rate (80%: 4 of 5 barcodes carry viral UMI) is misleading — two of
those four (bc2, bc4) are empty droplets with ambient reads. Over the 2 called cells
the rate is **100%**, the biologically meaningful number.

In [ ]:
# The per-cell table flags which barcodes are in the called-cell denominator.
per_cell[['barcode', 'virus_name', 'viral_umi', 'total_umi', 'is_called_cell']]

## In a real run

```bash
# Default: knee. Or emptyDrops, or an external CellRanger/STARsolo cell list:
viralscan -t t2g.txt -i index.idx -o out/ -s1 R1.fastq.gz -s2 R2.fastq.gz \
          --cell-calling external --called-cells-file cellranger_barcodes.tsv
```

**Always report `pct_infected_called`**, and state the denominator and UMI threshold
when comparing to published rates.